# Purpose:
- Generate czstack id - hcr id matching table from the last landmarks file

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import os
import json
DATA_DIR = Path('/root/capsule/data/')

In [2]:
def get_czstack_id(x):
    if x.startswith('cz'):
        return int(x.split('-')[0][2:])
    else:
        return -1


def get_hcr_id(x):
    if x.startswith('cz'):
        return int(x.split('-')[1][3:])
    else:
        return -1

In [7]:
subject_id = 788406
czstack_date = '2025-07-18'
save_dir = Path(f'/root/capsule/scratch/{subject_id}_{czstack_date}_coreg_cpsam/')

final_iter_num = 4
final_landmarks_file = save_dir / f'{subject_id}_{czstack_date}_landmarks_matched_ext_iter{final_iter_num}_qced.csv'

In [14]:
save_fn = save_dir / f'{subject_id}_coreg_table.csv'

final_qced_landmarks = pd.read_csv(final_landmarks_file)
columns = ['ids', 'active', 'czstack_x', 'czstack_y', 'czstack_z', 'hcr_x', 'hcr_y', 'hcr_z']
final_qced_landmarks.columns = columns
prev_qced_ids = final_qced_landmarks[final_qced_landmarks.ids.str.startswith('qced')].ids.values
prev_qced_ids = [qcid.replace('qced_','') for qcid in prev_qced_ids]
final_qced_landmarks.loc[final_qced_landmarks.ids.str.startswith('qced'), 'active'] = True
final_qced_landmarks.loc[final_qced_landmarks.ids.str.startswith('qced'), 'ids'] = prev_qced_ids
num_roi_points = sum(final_qced_landmarks.ids.str.startswith('cz'))

matched_landmarks = final_qced_landmarks.query('active').copy()

matched_landmarks['czstack_id'] = matched_landmarks.ids.apply(get_czstack_id)
matched_landmarks['hcr_id'] = matched_landmarks.ids.apply(get_hcr_id)
matched_landmarks = matched_landmarks[matched_landmarks.czstack_id != -1]
assert matched_landmarks.active.sum() == len(matched_landmarks)
assert len(set(matched_landmarks.czstack_id)) == len(matched_landmarks)
assert len(set(matched_landmarks.hcr_id)) == len(matched_landmarks)

matched_landmarks = matched_landmarks.sort_values(by='czstack_id')[['czstack_id', 'hcr_id']].reset_index(drop=True)
matched_landmarks.to_csv(save_fn, index=False)


print(f'Total number of czstack ids: {num_roi_points}')
print(f'Number of matched landmarks: {len(matched_landmarks)}')
print(f'Matching rate: {len(matched_landmarks)/num_roi_points:.2%}')

Total number of czstack ids: 931
Number of matched landmarks: 787
Matching rate: 84.53%


In [10]:
landmarks

,czstack_id,hcr_id
0,4.0,4817.0
1,5.0,3837.0
2,7.0,5522.0
3,8.0,5331.0
4,9.0,4818.0
...,...,...
782,921.0,120814.0
783,923.0,119515.0
784,924.0,123420.0
785,927.0,123989.0


In [3]:
mouse_id = 767018
save_dir = Path(f'/root/capsule/scratch/{mouse_id}_coreg')
qced_files = save_dir.glob(f'{mouse_id}_landmarks_matched_*_qced.csv')
max_iternums = np.max([int(fn.name.split('_iter')[1].split('_')[0]) for fn in qced_files])
print(max_iternums)


3


In [ ]:
for iternum in range(1, max_iternums+1):
    save_fn = save_dir / f'{mouse_id}_coreg_table_iter{iternum}.csv'
    if save_fn.exists():
        print(f'Coreg table from iter {iternum} for {mouse_id} exists. Skip...')
    else:
        filename = next(save_dir.glob(f'{mouse_id}_landmarks_matched_*iter{iternum}_*qced.csv'))
        landmarks = pd.read_csv(filename)
        columns = ['ids', 'active', 'czstack_x', 'czstack_y', 'czstack_z', 'hcr_x', 'hcr_y', 'hcr_z']
        landmarks.columns = columns


        landmarks['czstack_id'] = landmarks.ids.apply(get_czstack_id)
        landmarks['hcr_id'] = landmarks.ids.apply(get_hcr_id)
        landmarks = landmarks[landmarks.czstack_id != -1]
        landmarks = landmarks.query('active')
        landmarks = landmarks.sort_values(by='czstack_id')[['czstack_id', 'hcr_id']].reset_index(drop=True)
        landmarks.to_csv(save_fn, index=False)

In [11]:
landmarks

,czstack_id,hcr_id
0,2,9906
1,3,9874
2,7,5957
3,8,7239
4,12,6407
...,...,...
565,676,103596
566,677,105352
567,678,102360
568,682,104705
